# Outliers Detection - Magic Glasses (MG)

Dedicated notebook for MG outlier detection only:
- `OUTLIER_MAGIC_GLASSES_PARTIAL` (MAD15 -> MAD10)
- `OUTLIER_MAGIC_GLASSES_COMPLETE` (MAD15 -> MAD10 -> seasonal5 -> seasonal3)

In [ ]:
# Parameters with safe defaults
if (!exists("ROOT_PATH")) ROOT_PATH <- "~/workspace"
if (!exists("CONFIG_FILE_NAME")) CONFIG_FILE_NAME <- "SNT_config.json"
# Partial step is always required by MG logic; complete step is optional.
RUN_MAGIC_GLASSES_PARTIAL <- TRUE
if (!exists("RUN_MAGIC_GLASSES_COMPLETE")) RUN_MAGIC_GLASSES_COMPLETE <- FALSE
if (!exists("DEVIATION_MAD15")) DEVIATION_MAD15 <- 15
if (!exists("DEVIATION_MAD10")) DEVIATION_MAD10 <- 10
if (!exists("DEVIATION_SEASONAL5")) DEVIATION_SEASONAL5 <- 5
if (!exists("DEVIATION_SEASONAL3")) DEVIATION_SEASONAL3 <- 3
if (!exists("SEASONAL_WORKERS")) SEASONAL_WORKERS <- 1
if (!exists("DEV_SUBSET")) DEV_SUBSET <- FALSE
if (!exists("DEV_SUBSET_ADM1_N")) DEV_SUBSET_ADM1_N <- 2

if (SEASONAL_WORKERS < 1) {
  stop("SEASONAL_WORKERS must be >= 1")
}

if (DEV_SUBSET_ADM1_N < 1) {
  stop("DEV_SUBSET_ADM1_N must be >= 1")
}

CODE_PATH <- file.path(ROOT_PATH, "code")
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DIR <- file.path(DATA_PATH, "dhis2", "outliers_imputation")
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_outliers_imputation_magic_glasses")
source(file.path(PIPELINE_PATH, "utils", "snt_dhis2_outliers_imputation_magic_glasses.r"))

mg_input <- prepare_magic_glasses_input(
  root_path = ROOT_PATH,
  config_file_name = CONFIG_FILE_NAME,
  run_complete = RUN_MAGIC_GLASSES_COMPLETE,
  seasonal_workers = SEASONAL_WORKERS,
  dev_subset = DEV_SUBSET,
  dev_subset_adm1_n = DEV_SUBSET_ADM1_N
)

setup_ctx <- mg_input$setup_ctx
config_json <- mg_input$config_json
COUNTRY_CODE <- mg_input$country_code
fixed_cols <- mg_input$fixed_cols
indicators_to_keep <- mg_input$indicators_to_keep
dhis2_routine <- mg_input$dhis2_routine
dhis2_routine_long <- mg_input$dhis2_routine_long

CODE_PATH <- setup_ctx$CODE_PATH
CONFIG_PATH <- setup_ctx$CONFIG_PATH
DATA_PATH <- setup_ctx$DATA_PATH
openhexa <- setup_ctx$openhexa

In [ ]:
# Helpers loaded from utils/snt_dhis2_outliers_imputation_magic_glasses.r
# - prepare_magic_glasses_input()
# - run_magic_glasses_outlier_detection()
# - export_magic_glasses_outputs()

In [ ]:
detection_result <- run_magic_glasses_outlier_detection(
  dhis2_routine_long = dhis2_routine_long,
  deviation_mad15 = DEVIATION_MAD15,
  deviation_mad10 = DEVIATION_MAD10,
  run_complete = RUN_MAGIC_GLASSES_COMPLETE,
  deviation_seasonal5 = DEVIATION_SEASONAL5,
  deviation_seasonal3 = DEVIATION_SEASONAL3,
  seasonal_workers = SEASONAL_WORKERS
)

flagged_outliers_mad15_mad10 <- detection_result$flagged_outliers_mad15_mad10
flagged_outliers_seasonal5_seasonal3 <- detection_result$flagged_outliers_seasonal5_seasonal3

In [ ]:
export_magic_glasses_outputs(
  dhis2_routine_long = dhis2_routine_long,
  flagged_outliers_mad15_mad10 = flagged_outliers_mad15_mad10,
  flagged_outliers_seasonal5_seasonal3 = flagged_outliers_seasonal5_seasonal3,
  run_complete = RUN_MAGIC_GLASSES_COMPLETE,
  dhis2_routine = dhis2_routine,
  fixed_cols = fixed_cols,
  indicators_to_keep = indicators_to_keep,
  output_dir = OUTPUT_DIR,
  country_code = COUNTRY_CODE
)